In [2]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import os
from tqdm import tqdm
PATH_SOURCE = Path("/home/vlitoux/Documents/play-diverse/data")
PATH_INPUT = Path("./input")
PATH_AIS_250_VESSELS_INPUT = PATH_INPUT / "ais_250_vessels"
PATH_AIS_5_VESSELS_INPUT = PATH_INPUT / "ais_5_vessels"
%load_ext autoreload
%autoreload 2

In [2]:
def get_memory_usages(df_input: pd.DataFrame, ascending=False, sort_col='size_KB'):
	"""
	Get a overview of a dataframe (category, number unics, and size in KB).
	Sort according to 'sort_col'
	"""
	m_usage = df_input.memory_usage(deep=True) / 1024
	# Concatenate with unique
	uniques_df = df_input.nunique()
	return (
		pd.concat([m_usage, uniques_df, df_input.dtypes], axis=1)
		.rename(columns={0: 'size_KB', 1: 'distinct_number', 2: 'type'})
		.sort_values(sort_col, ascending=ascending)
	)

# Ship navi sim study

In [3]:
import gymnasium as gym
from ship_env import ShipEnvironment
from split_chunks import seperate_chunks
from ship_env import ShipEnvironment, chunk_to_traj
from datetime import timedelta
import utils
import pandas as pd
import polars as pl
import numpy as np
import gymnasium as gym
import datetime
import minari
import glob

# Prepare trajectories, times and overlap_idx (see data preparation section)
import pyrallis
plan_start_time = 0
plan_end_time = 24  
region_of_interest = {"LON": (103.82, 103.88), "LAT": (1.15, 1.22)}
region_of_interest_array = np.array([list(region_of_interest[k]) for k in ["LON", "LAT"]]).T
data_folder = Path("./raw_data/") # Read position data from multiple files
position_file_pattern = data_folder / "synthetic_ais_data.csv"
SHIP_ID_COL = "SHIP_ID"
TIME_UTC_COL = "TIMESTAMP_UTC"
SPEED_COL = "SPEED_KNOTSX10"
HEADING_COL = "HEADING"
LAT_COL = "LAT"
LON_COL = "LON"
X_COL = "X"
Y_COL = "Y"
CHUNK_ID_COL = "chunk_id"
position_file_pattern.exists()

True

In [4]:
df = pl.scan_csv(position_file_pattern, 
                        separator=";", 
                        null_values=["","NULL"], 
                        try_parse_dates=True).unique(subset=[SHIP_ID_COL, TIME_UTC_COL]).sort(by=[SHIP_ID_COL,TIME_UTC_COL]).collect()        
 # Remove duplicates based on SHIP_ID and TIMESTAMP_UTC
df


SHIP_ID,TIMESTAMP_UTC,LAT,LON,STATUS,HEADING,SPEED_KNOTSX10
i64,datetime[μs],f64,f64,i64,f64,f64
549,2017-08-05 23:03:40,1.202073,103.841191,0,107.242857,58.171429
549,2017-08-05 23:03:50,1.202167,103.841443,0,107.398654,58.013629
549,2017-08-05 23:04:00,1.202263,103.841693,0,107.547251,57.874251
549,2017-08-05 23:04:10,1.202364,103.841942,0,107.697905,58.146833
549,2017-08-05 23:04:20,1.202466,103.842192,0,107.819503,58.206915
…,…,…,…,…,…,…
5344,2018-06-17 23:54:10,1.217362,103.866254,0,125.086303,80.866482
5344,2018-06-17 23:54:20,1.217553,103.86658,0,125.292165,81.588304
5344,2018-06-17 23:54:30,1.217745,103.866908,0,125.497173,82.182575


In [5]:
threshold = timedelta(minutes=30)

# Create a new column for the time difference
# Equivalent to do a shift()
df = df.with_columns([
    pl.col(TIME_UTC_COL).diff().dt.cast_time_unit('ms').alias('time_diff'),
    pl.col(SHIP_ID_COL).diff().alias('ship_id_diff')
])

# Create a chunk_id column if
# - Different ship ID
# - A seperation threshold of at least 1800 seconds (30 minutes)
df = df.with_columns([
    pl.when((pl.col('time_diff') > threshold) | (pl.col('ship_id_diff') != 0))
    .then(1)
    .otherwise(0)
    .cum_sum()
    .alias(CHUNK_ID_COL)
])
df

SHIP_ID,TIMESTAMP_UTC,LAT,LON,STATUS,HEADING,SPEED_KNOTSX10,time_diff,ship_id_diff,chunk_id
i64,datetime[μs],f64,f64,i64,f64,f64,duration[ms],i64,i32
549,2017-08-05 23:03:40,1.202073,103.841191,0,107.242857,58.171429,null,null,0
549,2017-08-05 23:03:50,1.202167,103.841443,0,107.398654,58.013629,10s,0,0
549,2017-08-05 23:04:00,1.202263,103.841693,0,107.547251,57.874251,10s,0,0
549,2017-08-05 23:04:10,1.202364,103.841942,0,107.697905,58.146833,10s,0,0
549,2017-08-05 23:04:20,1.202466,103.842192,0,107.819503,58.206915,10s,0,0
…,…,…,…,…,…,…,…,…,…
5344,2018-06-17 23:54:10,1.217362,103.866254,0,125.086303,80.866482,10s,0,28
5344,2018-06-17 23:54:20,1.217553,103.86658,0,125.292165,81.588304,10s,0,28
5344,2018-06-17 23:54:30,1.217745,103.866908,0,125.497173,82.182575,10s,0,28


In [6]:
inter_pol = timedelta(seconds=10)
min_num_samples = 5

# Group by chunk_id and aggregate
chunks = df.group_by('chunk_id').agg([
    pl.col(SHIP_ID_COL).first().alias(SHIP_ID_COL),
    pl.col(TIME_UTC_COL).min().alias('start_time'),
    pl.col(TIME_UTC_COL).max().alias('end_time'),
    pl.count(TIME_UTC_COL).alias('num_records'),
    pl.struct(pl.all()).alias('all_records') # Note : this is a structing with the labels inside
]).sort([SHIP_ID_COL, 'start_time'])

chunks = chunks.filter(pl.col('num_records')>=min_num_samples)
chunks = chunks.sort('start_time')
chunks


chunk_id,SHIP_ID,start_time,end_time,num_records,all_records
i32,i64,datetime[μs],datetime[μs],u32,list[struct[10]]
0,549,2017-08-05 23:03:40,2017-08-05 23:30:10,160,"[{549,2017-08-05 23:03:40,1.202073,103.841191,0,107.242857,58.171429,null,null,0}, {549,2017-08-05 23:03:50,1.202167,103.841443,0,107.398654,58.013629,10s,0,0}, … {549,2017-08-05 23:30:10,1.219328,103.870815,0,139.903667,25.474547,10s,0,0}]"
1,550,2017-08-05 23:03:40,2017-08-05 23:09:20,35,"[{550,2017-08-05 23:03:40,1.195064,103.845519,0,289.937505,99.187503,-26m -30s,1,1}, {550,2017-08-05 23:03:50,1.194877,103.845095,0,289.945892,100.200065,10s,0,1}, … {550,2017-08-05 23:09:20,1.188711,103.830846,0,289.407796,102.68316,10s,0,1}]"
2,551,2017-08-05 23:03:40,2017-08-05 23:16:00,75,"[{551,2017-08-05 23:03:40,1.199706,103.858973,0,294.042858,90.028573,-5m -40s,1,2}, {551,2017-08-05 23:03:50,1.199511,103.858604,0,293.936553,90.143731,10s,0,2}, … {551,2017-08-05 23:16:00,1.186349,103.828128,0,290.184621,94.931966,10s,0,2}]"
3,552,2017-08-05 23:03:40,2017-08-05 23:16:20,77,"[{552,2017-08-05 23:03:40,1.207871,103.871991,0,297.999997,134.383084,-12m -20s,1,3}, {552,2017-08-05 23:03:50,1.207589,103.871435,0,297.813505,134.622326,10s,0,3}, … {552,2017-08-05 23:16:20,1.189779,103.82765,0,288.226215,114.818193,10s,0,3}]"
4,553,2017-08-05 23:03:40,2017-08-05 23:25:40,133,"[{553,2017-08-05 23:03:40,1.194016,103.820261,0,40.342868,109.428568,-12m -40s,1,4}, {553,2017-08-05 23:03:50,1.194601,103.820604,0,40.632714,146.566112,10s,0,4}, … {553,2017-08-05 23:25:40,1.191218,103.879566,0,93.312304,130.472603,10s,0,4}]"
…,…,…,…,…,…
24,5340,2018-06-17 23:27:00,2018-06-17 23:31:00,25,"[{5340,2018-06-17 23:27:00,1.185522,103.86058,0,115.0,128.0,-14m -40s,1,24}, {5340,2018-06-17 23:27:10,1.185763,103.861125,0,115.061711,128.807301,10s,0,24}, … {5340,2018-06-17 23:31:00,1.191802,103.875583,0,117.481203,156.419539,10s,0,24}]"
25,5341,2018-06-17 23:27:00,2018-06-17 23:32:50,36,"[{5341,2018-06-17 23:27:00,1.207136,103.838803,0,263.587496,112.537496,-4m,1,25}, {5341,2018-06-17 23:27:10,1.207192,103.83828,0,263.05011,113.535431,10s,0,25}, … {5341,2018-06-17 23:32:50,1.210355,103.821875,0,245.548638,44.431393,10s,0,25}]"
26,5342,2018-06-17 23:27:00,2018-06-17 23:40:30,82,"[{5342,2018-06-17 23:27:00,1.204917,103.871541,0,295.999994,128.484471,-5m -50s,1,26}, {5342,2018-06-17 23:27:10,1.204672,103.870996,0,295.949232,129.091979,10s,0,26}, … {5342,2018-06-17 23:40:30,1.186496,103.829258,0,288.44168,117.917194,10s,0,26}]"


In [7]:
def lon_to_xpos(lon, origin_lon, origin_lat):
    return utils.haversine_distance(origin_lon, origin_lat, lon, origin_lat)    

def lat_to_ypos(lat, origin_lon, origin_lat):
    return utils.haversine_distance(origin_lon, origin_lat, origin_lon, lat)

def knots_to_ms(speed_knots):
    """Convert speed from 10xknots to meters per second"""
    return speed_knots * 0.0514444

def degrees_to_radians(degrees):
    """Convert angles from degrees to radians"""
    return np.radians(degrees)

def radians_2d_to_heading(radians_2d):
    
    # Reverse the coordinate system adjustment
    heading_radians = -1 * radians_2d + np.pi/2
    
    # Convert to degrees
    heading_degrees = np.degrees(heading_radians)
    
    # Normalize to range [0, 360)
    normalized_degrees = heading_degrees % 360
    
    return normalized_degrees
def heading_to_2d_radians(heading_degrees):
    # Convert to radians
    heading_radians = np.radians(heading_degrees)
    
    # Adjust for coordinate system difference
    adjusted_radians = -1 * (heading_radians - np.pi/2)
    
    # Normalize to range [0, 2π)
    normalized_radians = adjusted_radians % (2 * np.pi)
    
    return normalized_radians

def df_to_trajs(df_in: pl.DataFrame, interpol_interval: timedelta, region_of_interest_array: np.ndarray):
    """
    Arrange and interpolate messages per chunk
    """
    id_chunk = df_in["chunk_id"][0]
    interpol_seconds = int(interpol_interval.total_seconds())
    if interpol_seconds != interpol_interval.total_seconds():
        raise ValueError("interpol_interval must be in integer seconds")
    # Find the start and end times, rounded to the nearest interpol_interval
    # start_time will uses ceil and end_time uses floor due to interpolation
    start_time = df_in[TIME_UTC_COL].min().replace(microsecond=0)
    # Can delete the first message by doing that
    start_time = start_time + timedelta(seconds=interpol_seconds - start_time.second % interpol_seconds)            
    end_time = df_in[TIME_UTC_COL].max().replace(microsecond=0)
    end_time = end_time - timedelta(seconds=end_time.second % interpol_seconds)    
    original_time = df_in[TIME_UTC_COL].map_elements(lambda x: int(x.timestamp()), return_dtype=pl.Int64)        

    # Define the origin as the southwest corner of the region of interest
    origin_lon, origin_lat = region_of_interest_array[0][0], region_of_interest_array[0][1]
    
    # Convert to numpy arrays for interpolation
    original_time_np = original_time.to_numpy()
    y_np = np.apply_along_axis(lambda x: lat_to_ypos(x, origin_lon, origin_lat), 0, df_in[LAT_COL].to_numpy())
    x_np = np.apply_along_axis(lambda x: lon_to_xpos(x, origin_lon, origin_lat), 0, df_in[LON_COL].to_numpy())
    speed_np = np.apply_along_axis(knots_to_ms, 0, df_in[SPEED_COL].to_numpy())
    heading_np = np.apply_along_axis(heading_to_2d_radians, 0, df_in[HEADING_COL].to_numpy())    
    # Arrange time in case of missing message.
    # Example : 10,30,60,90 -> 10, 20, 30, 40, 50, 60, 70, 80, 90
    t_np = np.arange(start_time.timestamp(), end_time.timestamp() + interpol_seconds, interpol_seconds, dtype=np.int64)
    # Interpolate the trajectory by using arranged time.
    traj_y = np.interp(t_np, original_time_np, y_np)    
    traj_x = np.interp(t_np, original_time_np, x_np)
    traj_speed = np.interp(t_np, original_time_np, speed_np)
    traj_heading = utils.shortest_path_angle_interp(t_np, original_time_np, heading_np)
    ret = pl.DataFrame([traj_x,traj_y,traj_speed,traj_heading,t_np],schema=[X_COL,Y_COL,SPEED_COL,HEADING_COL,TIME_UTC_COL])
    ret = ret.with_columns(pl.lit(id_chunk).alias(CHUNK_ID_COL))
    return ret

df_ret = df.group_by(CHUNK_ID_COL).map_groups(lambda x: df_to_trajs(x, inter_pol, region_of_interest_array))
df_ret

X,Y,SPEED_KNOTSX10,HEADING,TIMESTAMP_UTC,chunk_id
f64,f64,f64,f64,i64,i32
4929.578125,4869.997559,6.064129,5.802005,1518996150,12
4983.295898,4898.20752,6.067429,5.800623,1518996160,12
5037.07373,4926.26709,6.065819,5.798833,1518996170,12
5090.776367,4954.136719,6.050374,5.797197,1518996180,12
5144.535644,4981.760742,6.04414,5.795509,1518996190,12
…,…,…,…,…,…
780.278809,4338.590332,5.703994,2.815766,1529271660,23
727.647156,4315.523438,5.746449,2.816508,1529271670,23
674.481384,4291.808105,5.821519,2.817126,1529271680,23


In [38]:
f_trajs = df_ret.group_by("chunk_id").agg(pl.concat_list([X_COL,Y_COL,SPEED_COL,HEADING_COL]).alias("trajs"),
                                          pl.col(TIME_UTC_COL),
                                          pl.col(TIME_UTC_COL).min().alias("min_time"),
                                          pl.col(TIME_UTC_COL).max().alias("max_time")
                                          ).sort("chunk_id")
f_trajs

chunk_id,trajs,TIMESTAMP_UTC,min_time,max_time
i32,list[list[f64]],list[i64],i64,i64
0,"[[2383.872803, 5800.695312, … 5.979522], [2411.656738, 5811.39502, … 5.976928], … [5649.264648, 7708.961426, … 5.412202]]","[1501967030, 1501967040, … 1501968610]",1501967030,1501968610
1,"[[2789.832031, 4990.087402, … 2.793471], [2742.337158, 4969.068848, … 2.793468], … [1205.764404, 4304.481934, … 2.802863]]","[1501967030, 1501967040, … 1501967360]",1501967030,1501967360
2,"[[4291.708984, 5505.394043, … 2.723821], [4250.589355, 5483.662109, … 2.725885], … [903.594116, 4041.879883, … 2.789305]]","[1501967030, 1501967040, … 1501967760]",1501967030,1501967760
3,"[[5718.148437, 6403.640625, … 2.656155], [5656.237793, 6372.437012, … 2.659175], … [850.511902, 4423.236816, … 2.823485]]","[1501967030, 1501967040, … 1501967780]",1501967030,1501967780
4,"[[67.157272, 4959.375977, … 0.861622], [118.206619, 4916.078613, … 0.850066], … [6622.14209, 4583.219727, … 6.225375]]","[1501967030, 1501967040, … 1501968340]",1501967030,1501968340
…,…,…,…,…
24,"[[4572.007324, 3976.614014, … 5.845776], [4633.290039, 4004.138672, … 5.844387], … [6179.300293, 4648.179687, … 5.803548]]","[1529270830, 1529270840, … 1529271060]",1529270830,1529271060
25,"[[2032.247192, 6359.469238, … 3.262891], [1973.19043, 6366.666992, … 3.272405], … [208.492035, 6711.156738, … 3.568349]]","[1529270830, 1529270840, … 1529271170]",1529270830,1529271170
26,"[[5669.383789, 6079.260254, … 2.688693], [5609.40625, 6051.558594, … 2.691439], … [1029.244141, 4058.131836, … 2.819725]]","[1529270830, 1529270840, … 1529271630]",1529270830,1529271630


In [54]:
# Get each other vessels that passes at the same time of the current vessel
overlap_idx = f_trajs.join(f_trajs[["chunk_id","min_time","max_time"]],how="cross").filter(
    (pl.col("min_time") >= pl.col("min_time_right")) & (pl.col("min_time") <= pl.col("max_time_right")) & (pl.col("chunk_id") != pl.col("chunk_id_right"))
    ).group_by("chunk_id").agg(pl.col("chunk_id_right")).sort("chunk_id")["chunk_id_right"].to_numpy()
overlap_idx

array([array([1, 2, 3, 4], dtype=int32), array([0, 2, 3, 4], dtype=int32),
       array([0, 1, 3, 4], dtype=int32), array([0, 1, 2, 4], dtype=int32),
       array([0, 1, 2, 3], dtype=int32),
       array([ 6,  7,  8,  9, 10, 11], dtype=int32),
       array([ 5,  7,  8,  9, 10, 11], dtype=int32),
       array([ 5,  6,  8,  9, 10, 11], dtype=int32),
       array([ 5,  6,  7,  9, 10, 11], dtype=int32),
       array([ 5,  6,  7,  8, 10, 11], dtype=int32),
       array([ 5,  6,  7,  8,  9, 11], dtype=int32),
       array([ 5,  6,  7,  8,  9, 10], dtype=int32),
       array([13, 14, 15, 16, 17], dtype=int32),
       array([12, 14, 15, 16, 17], dtype=int32),
       array([12, 13, 15, 16, 17], dtype=int32),
       array([12, 13, 14, 16, 17], dtype=int32),
       array([12, 13, 14, 15, 17], dtype=int32),
       array([12, 13, 14, 15, 16], dtype=int32),
       array([19, 20, 21], dtype=int32), array([18, 20, 21], dtype=int32),
       array([18, 19, 21], dtype=int32), array([18, 19, 20], dtype=in

In [105]:
#Create environment
max_size = max([len(val) for val in overlap_idx])  #Useless  
env = ShipEnvironment(f_trajs["trajs"].to_numpy(), f_trajs[TIME_UTC_COL].to_numpy(), overlap_idx, region_of_interest, n_neighbor_agents=10)
env = gym.wrappers.RecordVideo(env, "videos/",name_prefix="shipNavi-",episode_trigger=lambda _: True) # Record each episode
env.metadata['render_fps'] = 10    
env

/home/victor/.pyenv/versions/geodata_venv/lib/python3.12/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /home/victor/Documents/ShipNaviSim-play/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


<RecordVideo<ShipEnvironment instance>>

In [107]:
def replay_ship_env(id_ship : int, env : ShipEnvironment):
    """
    Reset the environmenet and run the simulation
    """
    list_action = []
    obs, info = env.reset(seed=42, options = {'ego_pos': id_ship})
    actions = info['actions']
    if(actions is None):
        return None
    for i in range(1000):        
        action = actions[i]
        list_action.append(action)
        observation, _, terminated, truncated, info = env.step(action)
        # print(i, observation)
        # print()
        if terminated or truncated:          
            break    
    return info,np.array(list_action) # to do stats

stats, list_action = replay_ship_env(id_ship=1, env=env)    
stats, list_action = replay_ship_env(id_ship=0, env=env)    
pl.DataFrame(list_action)

Moviepy - Building video /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-0.mp4.
Moviepy - Writing video /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-0.mp4



Moviepy - Done !
Moviepy - video ready /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-0.mp4
Moviepy - Building video /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-1.mp4.
Moviepy - Writing video /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-1.mp4



Moviepy - Done !
Moviepy - video ready /home/victor/Documents/ShipNaviSim-play/videos/shipNavi--episode-1.mp4


column_0,column_1,column_2
f32,f32,f32
27.783936,10.699707,-0.002594
27.710693,11.266113,-0.002629
27.741943,11.271484,-0.002122
27.56665,11.30127,-0.001619
27.268311,11.59082,-0.00206
…,…,…
2.921387,2.620605,-0.004989
4.424805,4.152344,-0.005053
7.503418,5.397461,-0.004814


### Load the model (train with "IL-BC.py")

In [2]:
import det_bc
import torch as th
from pathlib import Path

# Reconstruct policy
PATH_POLICIES = Path("ckpoints/BC-deterministic-256-128hid-256batch-combMLPTanh-Maritime-Expert-v1.th")
policies =  th.load(PATH_POLICIES, map_location="cpu",weights_only=False)
policies


DetPolicy(
  (features_extractor): NewCombinedNormExtractor(
    (extractors): ModuleDict(
      (ego): Sequential(
        (0): Sequential(
          (ego-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (ego-net_normalize_input): RunningNorm()
          (ego-net_dense_final): Linear(in_features=44, out_features=64, bias=True)
        )
        (1): Tanh()
      )
      (goal): Sequential(
        (0): Sequential(
          (goal-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (goal-net_normalize_input): RunningNorm()
          (goal-net_dense_final): Linear(in_features=2, out_features=8, bias=True)
        )
        (1): Tanh()
      )
      (neighbors): Sequential(
        (0): Sequential(
          (neighbors-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (neighbors-net_normalize_input): RunningNorm()
          (neighbors-net_dense0): Linear(in_features=440, out_features=256, bias=True)
          (neighbors-net_act0): ReLU()
          (neighbors-net_dense

In [3]:
policies.eval()

DetPolicy(
  (features_extractor): NewCombinedNormExtractor(
    (extractors): ModuleDict(
      (ego): Sequential(
        (0): Sequential(
          (ego-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (ego-net_normalize_input): RunningNorm()
          (ego-net_dense_final): Linear(in_features=44, out_features=64, bias=True)
        )
        (1): Tanh()
      )
      (goal): Sequential(
        (0): Sequential(
          (goal-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (goal-net_normalize_input): RunningNorm()
          (goal-net_dense_final): Linear(in_features=2, out_features=8, bias=True)
        )
        (1): Tanh()
      )
      (neighbors): Sequential(
        (0): Sequential(
          (neighbors-net_flatten): Flatten(start_dim=1, end_dim=-1)
          (neighbors-net_normalize_input): RunningNorm()
          (neighbors-net_dense0): Linear(in_features=440, out_features=256, bias=True)
          (neighbors-net_act0): ReLU()
          (neighbors-net_dense

In [ ]:
#Create minari dataset for IL
# Uncomment for creating offline dataset. Note that, we only need to create dataset once.
name_dataset = "Maritime-Expert-v1"
if name_dataset not in minari.list_local_datasets().keys():
    create_minari_dataset(env, dataset_name=name_dataset,num_ships=num_ships)
dataset = minari.load_dataset(name_dataset)
dataset

In [ ]:
for episode_data in dataset.iterate_episodes():
        observations = episode_data.observations
        actions = episode_data.actions
        rewards = episode_data.rewards
        terminations = episode_data.terminations
        truncations = episode_data.truncations
        infos = episode_data.infos

In [ ]:
lst_infos = []
for id_ship in range(num_ships):
    val = replay_ship_env(id_ship, env)
    if(val is None):
        continue
    val["ship_id"] = chunks[id_ship]["SHIP_ID"][0]
    val["start_time"] = chunks[id_ship]["start_time"][0]
    lst_infos.append(val)
df_exp_infos = pd.DataFrame(lst_infos)
df_exp_infos.drop(["CPD"], axis=1, inplace=True)
df_exp_infos.head()

In [ ]:
df_exp_infos.to_csv(f"exp_stats_{plan_start_time}to{plan_end_time}.csv", sep=";", index=False) 